In [ ]:
from cst.datos import cargar_caudal_genil
from cst.datos import cargar_lluvia_genil
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

caudal = cargar_caudal_genil()
lluvia = cargar_lluvia_genil()

In [ ]:
df = pd.DataFrame({"caudal": caudal, "lluvia": lluvia}).dropna()

In [ ]:
def construir_features(df, target="caudal", exog="lluvia"):
    """Devuelve X (DataFrame) e y (Series) listos para fit/predict."""
    out = pd.DataFrame(index=df.index)
    
    for lag in (1, 2, 3, 365):
        out[f"q_lag{lag}"] = df[target].shift(lag)

    for w in (3,):
        out[f"p_acum{w}d"] = df[exog].rolling(w).sum().shift(1)

    idx = out.index
    out["sin_mes"] = np.sin(2 * np.pi * idx.month / 12)
    out["cos_mes"] =  np.cos(2 * np.pi * idx.month / 12)
    out["sin_an"] = np.sin(2 * np.pi * idx.dayofyear / 365.25)
    out["cos_an"] = np.cos(2 * np.pi * idx.dayofyear / 365.25)

    out["y"] = df[target]

    # dropna() pierde el atributo freq → lo reconstruimos con asfreq('D')
    return out

In [ ]:
df = construir_features(df).dropna()

In [ ]:
train = df[:"2022-12-31"]
val = df["2023-01-01":"2024-12-31"]
test = df["2025-01-01":]

In [ ]:
train.index.min(), train.index.max()

In [ ]:
val.index.min(), val.index.max()

In [ ]:
test.index.min(), test.index.max()

In [ ]:
X_train, y_train = train.drop(columns="y"), train["y"]
X_val, y_val = val.drop(columns="y"), val["y"]
X_test, y_test = test.drop(columns="y"), test["y"]

In [ ]:
from itertools import product

n_estimators_grid = [500, 1000, 2000]
max_depth_grid = [3, 5, None]
max_features_grid = [0.5, 0.75, 1]

grid = product(n_estimators_grid, max_depth_grid, max_features_grid)

In [ ]:
res_full = []
for n_estimators, max_depth, max_features in grid:
    res = {
        "n_estimators": n_estimators, 
        "max_depth": max_depth, 
        "max_features": max_features
    }
    
    model = RandomForestRegressor(
        n_estimators=n_estimators, 
        max_depth=max_depth, 
        max_features=max_features
    )
    model.fit(X_train, y_train)
    y_pred_val = model.predict(X_val)
    res["mae"] = mean_absolute_error(y_val, y_pred_val)
    res_full.append(res)

In [ ]:
pd.DataFrame(res_full).sort_values("mae").head(5)

In [ ]:
X_trval = pd.concat((X_train, X_val))
y_trval = pd.concat((y_train, y_val))

In [ ]:
best_model = RandomForestRegressor(
    n_estimators=1000, 
    max_depth=None, 
    max_features=0.75
)
best_model.fit(X_trval, y_trval)
y_pred_test = best_model.predict(X_test)

In [ ]:
pred = pd.DataFrame({"y_test": y_test, "y_pred_val": y_pred_test})

In [ ]:
pred.plot()

In [ ]:
mean_absolute_error(y_test, y_pred_test)

In [ ]:
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV

In [ ]:
param_grid = {
    "n_estimators": n_estimators_grid, 
    "max_depth": max_depth_grid, 
    "max_features": max_features_grid
}

ts = TimeSeriesSplit(n_splits=5)
cv = GridSearchCV(RandomForestRegressor(), param_grid=param_grid, cv=ts)

In [ ]:
cv.fit(X_trval, y_trval)

In [ ]:
pd.DataFrame(cv.cv_results_).sort_values("rank_test_score").head(5)

In [ ]:
y_pred_test = cv.predict(X_test)

In [ ]:
pred = pd.DataFrame({"y_test": y_test, "y_pred_val": y_pred_test})

In [ ]:
pred.plot()